# Stage 3: Data Extraction
Capstone Project: Spam Message Analysis Using Malicious URLs

Parse each URL (text only, URLs are never opened) and extract structural and lexical features such as length, domain, TLD, HTTPS, IP address, special characters and suspicious keywords.

In [1]:
!pip -q install tldextract

In [2]:
from pathlib import Path

# Project root = parent of this notebook's stage folder (run the notebook from inside its own folder)
ROOT = Path.cwd().parent
RAW_CSV = ROOT / "dataset" / "dataset.csv"
INTER = ROOT / "dataset" / "intermediate"      # output of each stage becomes input of the next
RESULTS = ROOT / "results"
GRAPH_DIR = RESULTS / "graphs"
for p in (INTER, RESULTS, GRAPH_DIR):
    p.mkdir(parents=True, exist_ok=True)


In [3]:
import pandas as pd
import numpy as np
import re
import ipaddress
from urllib.parse import urlparse
import tldextract

df = pd.read_csv(INTER / "stage2_filtered.csv", keep_default_na=False)
print("Loaded from Stage 2:", df.shape)


Loaded from Stage 2: (651066, 2)


## Stage 3 — Data Extraction
Parse each URL and extract structural/lexical features (domain, TLD, lengths, counts, suspicious keywords, etc.).

In [4]:
# Uses an offline suffix list; URLs are never opened/fetched.
extractor = tldextract.TLDExtract(suffix_list_urls=None)

shorteners = {
    "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly",
    "is.gd", "buff.ly", "rebrand.ly", "cutt.ly",
    "shorturl.at", "tiny.cc"
}

suspicious_words = [
    "login", "verify", "secure", "account", "bank",
    "update", "free", "confirm", "password", "signin",
    "wallet", "payment", "bonus", "support"
]

def add_scheme(url):
    """Adds http:// only for safe text parsing when a scheme is missing."""
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", url):
        return "http://" + url
    return url

def get_parts(url):
    full_url = add_scheme(url)

    try:
        parsed = urlparse(full_url)
        hostname = (parsed.hostname or "").lower()
        path = parsed.path
        query = parsed.query
        scheme = parsed.scheme.lower()
    except ValueError:
        # Fallback for malformed URLs (e.g. invalid IPv6 brackets)
        hostname, path, query, scheme = "", "", "", ""

    ext = extractor(hostname) if hostname else extractor("")

    domain = f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain
    subdomain_count = len([x for x in ext.subdomain.split(".") if x]) if ext.subdomain else 0

    has_ip = 0
    if hostname:
        try:
            ipaddress.ip_address(hostname)
            has_ip = 1
        except ValueError:
            has_ip = 0

    extension_match = re.search(r"\.([a-zA-Z0-9]{1,8})$", path)
    file_extension = "." + extension_match.group(1).lower() if extension_match else ""

    url_lower = url.lower()
    found_words = [word for word in suspicious_words if word in url_lower]
    special_char_count = len(re.findall(r"[^a-zA-Z0-9]", url))

    return pd.Series({
        "url_length": len(url),
        "domain": domain,
        "tld": ext.suffix,
        "subdomain_count": subdomain_count,
        "path_length": len(path),
        "query_length": len(query),
        "has_https": int(scheme == "https"),
        "has_ip_address": has_ip,
        "dot_count": url.count("."),
        "hyphen_count": url.count("-"),
        "digit_count": sum(ch.isdigit() for ch in url),
        "special_char_count": special_char_count,
        "has_at_symbol": int("@" in url),
        "has_shortener": int(hostname in shorteners),
        "suspicious_keyword": ", ".join(found_words) if found_words else "none",
        "keyword_count": len(found_words),
        "file_extension": file_extension,
    })

features = df["url"].apply(get_parts)
df_final = pd.concat([df.reset_index(drop=True), features.reset_index(drop=True)], axis=1)

print("Extracted feature columns:", features.columns.tolist())
df_final.head()

Extracted feature columns: ['url_length', 'domain', 'tld', 'subdomain_count', 'path_length', 'query_length', 'has_https', 'has_ip_address', 'dot_count', 'hyphen_count', 'digit_count', 'special_char_count', 'has_at_symbol', 'has_shortener', 'suspicious_keyword', 'keyword_count', 'file_extension']


,url,type,url_length,domain,tld,subdomain_count,path_length,query_length,has_https,has_ip_address,dot_count,hyphen_count,digit_count,special_char_count,has_at_symbol,has_shortener,suspicious_keyword,keyword_count,file_extension
0,usa-people-search.com/Find-Jason-Bowser.aspx,benign,44,usa-people-search.com,com,0,23,0,0,0,2,4,0,7,0,0,none,0,.aspx
1,http://huangjintawujin.cn/examples/jsp/jsp2/js...,malware,74,huangjintawujin.cn,cn,0,49,0,0,0,2,0,1,10,0,0,none,0,.html
2,kellysconfederateirishbrigade.com/battles/miss...,benign,55,kellysconfederateirishbrigade.com,com,0,22,0,0,0,2,0,0,4,0,0,none,0,.html
3,http://www.easylivinggreenproducts.com.au/shop...,defacement,83,easylivinggreenproducts.com.au,com.au,1,10,31,0,0,5,0,2,14,0,0,none,0,.html
4,heavymetalmusicnews.com/,benign,24,heavymetalmusicnews.com,com,0,1,0,0,0,1,0,0,2,0,0,none,0,


In [5]:
df_final.to_csv(INTER / "stage3_features.csv", index=False)
print("Saved:", INTER / "stage3_features.csv")

Saved: C:\Users\karth\Desktop\spam-message-analysis-malicious-urls\dataset\intermediate\stage3_features.csv
